# Synthetic Engine Generator (C-MAPSS-based)

Generates fake, streaming sensor data for a synthetic turbofan engine, using the
statistical structure learned from real C-MAPSS engines. Rules carried over from the
PHM2010 synthetic-blade work:

1. **No circularity**: synthetic engines are used only to feed the live dashboard
   stream — model performance is always reported using real engines only (Sections
   6-18 of the main notebook).
2. **Parametric bootstrap, not independent resampling**: each synthetic engine is
   based on one real engine's trajectory shape plus controlled noise, preserving
   realistic parameter correlations rather than mixing independent random values.
3. **Plausibility filtering**: generated engines with implausible lifespans or sensor
   ranges (outside what's observed in real data) are rejected.

## 1. Load Real Engine Trajectories as Templates

In [22]:
import pandas as pd
import numpy as np
import os
import pickle
import json

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', 'cmapss')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

col_names = (['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3']
             + [f's_{i}' for i in range(1, 22)])
train = pd.read_csv(os.path.join(DATA_DIR, 'train_FD001.txt'), sep=r'\s+', header=None, names=col_names)

with open(os.path.join(MODELS_DIR, 'cmapss_metadata.json')) as f:
    metadata = json.load(f)

active_sensors = ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']

print(f"Loaded {train['unit_number'].nunique()} real engines as templates")
print(f"Lifespan range: {train.groupby('unit_number')['time_in_cycles'].max().min()} - {train.groupby('unit_number')['time_in_cycles'].max().max()} cycles")

Loaded 100 real engines as templates
Lifespan range: 128 - 362 cycles


## 2. Parametric Bootstrap: One Synthetic Engine at a Time

Pick a random real engine as the "template", stretch/compress its lifespan slightly,
and add small proportional noise per sensor — preserving the template's overall
shape and inter-sensor relationships rather than generating each sensor independently.

In [23]:
def generate_synthetic_engine(train_df, sensors, life_stretch_std=0.10, noise_std=0.02, seed=None):
    rng = np.random.default_rng(seed)

    template_unit = rng.choice(train_df['unit_number'].unique())
    template = train_df[train_df['unit_number'] == template_unit].sort_values('time_in_cycles').reset_index(drop=True)

    stretch = 1.0 + rng.normal(0, life_stretch_std)
    stretch = np.clip(stretch, 0.6, 1.6)
    n_cycles = max(int(len(template) * stretch), 30)

    old_idx = np.linspace(0, len(template) - 1, len(template))
    new_idx = np.linspace(0, len(template) - 1, n_cycles)

    synth = pd.DataFrame({'time_in_cycles': np.arange(1, n_cycles + 1)})
    for s in sensors:
        interpolated = np.interp(new_idx, old_idx, template[s].values)
        noise = rng.normal(0, noise_std * (template[s].std() + 1e-6), size=n_cycles)
        synth[s] = interpolated + noise

    return synth, template_unit

test_engine, src = generate_synthetic_engine(train, active_sensors, seed=42)
print(f"Generated engine based on real unit {src}, lifespan = {len(test_engine)} cycles")
test_engine.head()

Generated engine based on real unit 9, lifespan = 180 cycles


,time_in_cycles,s_2,s_3,s_4,s_7,s_8,s_9,s_11,s_12,s_13,s_14,s_15,s_17,s_20,s_21
0,1,642.187185,1582.939817,1396.393594,554.353396,2387.957164,9065.472175,47.092026,522.349771,2387.948144,8149.663904,8.401788,391.001022,39.044286,23.530573
1,2,641.988223,1581.173259,1394.761408,554.835811,2387.948697,9063.977612,47.120290,522.977582,2387.990132,8148.152248,8.429259,391.874381,38.986164,23.432645
2,3,642.373388,1586.939567,1399.023156,554.849766,2388.021813,9058.112834,47.101746,522.681012,2388.040422,8142.116097,8.396130,391.208754,39.061490,23.451559
3,4,642.034572,1583.207438,1404.446938,555.064633,2387.984219,9063.644593,47.076802,522.728917,2387.999828,8146.338256,8.370050,391.286544,39.006343,23.490664
4,5,641.957984,1584.485170,1400.554023,554.645694,2387.978508,9062.203036,47.105909,522.513812,2388.011260,8145.312682,8.361571,391.414354,38.953174,23.434893


## 3. Plausibility Filter

Reject synthetic engines whose lifespan or sensor ranges fall outside what real
engines ever showed — mirroring the PHM2010 ceiling check that caught an
unrealistically fast-degrading synthetic blade.

In [24]:
real_lifespans = train.groupby('unit_number')['time_in_cycles'].max()
LIFE_MIN, LIFE_MAX = real_lifespans.min() * 0.8, real_lifespans.max() * 1.2

real_ranges = {s: (train[s].min(), train[s].max()) for s in active_sensors}

def is_plausible(synth_df, sensors):
    if not (LIFE_MIN <= len(synth_df) <= LIFE_MAX):
        return False
    for s in sensors:
        lo, hi = real_ranges[s]
        margin = (hi - lo) * 0.3
        if synth_df[s].min() < lo - margin or synth_df[s].max() > hi + margin:
            return False
    return True

print("Test engine plausible?", is_plausible(test_engine, active_sensors))
print(f"Allowed lifespan range: {LIFE_MIN:.0f} - {LIFE_MAX:.0f} cycles")

Test engine plausible? True
Allowed lifespan range: 102 - 434 cycles


## 4. Batch Generation with Rejection Sampling

In [25]:
def generate_batch(train_df, sensors, n_engines=30, max_attempts=200, seed=0):
    rng_seed = seed
    accepted = []
    attempts = 0
    while len(accepted) < n_engines and attempts < max_attempts:
        synth, src = generate_synthetic_engine(train_df, sensors, seed=rng_seed)
        if is_plausible(synth, sensors):
            synth['synthetic_id'] = f'synth_{len(accepted):03d}'
            synth['source_unit'] = src
            accepted.append(synth)
        rng_seed += 1
        attempts += 1
    print(f"Accepted {len(accepted)}/{attempts} attempts")
    return accepted

synthetic_engines = generate_batch(train, active_sensors, n_engines=30)

Accepted 30/30 attempts


## 5. Live Streaming Interface

The dashboard needs to simulate a "ticking" sensor feed — one new cycle at a time,
automatically starting a new synthetic engine when the current one "fails" (reaches
its generated lifespan). This class wraps the generator into a stateful stream.

In [26]:
class SyntheticEngineStream:
    def __init__(self, train_df, sensors, feature_cols, model, seed=0):
        self.train_df = train_df
        self.sensors = sensors
        self.feature_cols = feature_cols
        self.model = model
        self.rng_seed = seed
        self._start_new_engine()

    def _start_new_engine(self):
        while True:
            synth, src = generate_synthetic_engine(self.train_df, self.sensors, seed=self.rng_seed)
            self.rng_seed += 1
            if is_plausible(synth, self.sensors):
                self.current_engine = synth
                self.current_source = src
                self.cycle_idx = 0
                break

    def tick(self):
        """Advance one cycle. Returns dict with sensor readings, engine metadata, and model prediction."""
        if self.cycle_idx >= len(self.current_engine):
            self._start_new_engine()

        window = self.current_engine.iloc[:self.cycle_idx + 1]
        row = window.iloc[-1].to_dict()

        # feature'ları anlık pencere üzerinden hesapla (rolling mean, trend, deviation)
        feats = {}
        for s in self.sensors:
            vals = window[s].values
            cycles = window['time_in_cycles'].values
            feats[f'{s}'] = vals[-1]
            feats[f'{s}_rm'] = vals[-10:].mean()
            feats[f'{s}_dev'] = vals[-1] - vals[0]
            feats[f'{s}_trend'] = np.polyfit(cycles, vals, 1)[0] if len(vals) > 1 else 0.0

        X = pd.DataFrame([feats])[self.feature_cols]
        proba = self.model.predict_proba(X)[0, 1]
        pred = int(proba >= 0.5)

        self.cycle_idx += 1

        return {
            'engine_source_unit': int(self.current_source),
            'cycle': int(row['time_in_cycles']),
            'engine_total_lifespan': len(self.current_engine),
            'sensors': {s: round(row[s], 3) for s in self.sensors},
            'maintenance_flag': pred,
            'maintenance_probability': round(float(proba), 3)
        }

## 6. Quick Test of the Stream

In [27]:
with open(os.path.join(MODELS_DIR, 'cmapss_model.pkl'), 'rb') as f:
    clf_loaded = pickle.load(f)

feature_cols = metadata['feature_columns']

stream = SyntheticEngineStream(train, active_sensors, feature_cols, clf_loaded)

for _ in range(15):
    tick_data = stream.tick()
    print(f"Engine (src {tick_data['engine_source_unit']}) | cycle {tick_data['cycle']}/{tick_data['engine_total_lifespan']} "
          f"| flag={tick_data['maintenance_flag']} | proba={tick_data['maintenance_probability']}")

Engine (src 86) | cycle 1/274 | flag=0 | proba=0.011
Engine (src 86) | cycle 2/274 | flag=0 | proba=0.094
Engine (src 86) | cycle 3/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 4/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 5/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 6/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 7/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 8/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 9/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 10/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 11/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 12/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 13/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 14/274 | flag=0 | proba=0.0
Engine (src 86) | cycle 15/274 | flag=0 | proba=0.0


In [28]:
stream2 = SyntheticEngineStream(train, active_sensors, feature_cols, clf_loaded, seed=100)

print("Motorun son 20 çevrimi (ömrünün sonuna doğru):\n")
history = []
for _ in range(300):  # bir motorun tamamını (ve belki yeni bir motorun başlangıcını) geçmek için yeterli
    tick_data = stream2.tick()
    history.append(tick_data)

history_df = pd.DataFrame(history)
first_engine_len = history_df['engine_total_lifespan'].iloc[0]
first_engine_history = history_df.iloc[:first_engine_len]

print(first_engine_history[['cycle', 'engine_total_lifespan', 'maintenance_flag', 'maintenance_probability']].tail(20).to_string(index=False))

Motorun son 20 çevrimi (ömrünün sonuna doğru):

 cycle  engine_total_lifespan  maintenance_flag  maintenance_probability
   139                    158                 1                    0.703
   140                    158                 1                    0.739
   141                    158                 1                    0.897
   142                    158                 1                    0.915
   143                    158                 1                    0.947
   144                    158                 1                    0.969
   145                    158                 1                    0.976
   146                    158                 1                    0.974
   147                    158                 1                    0.971
   148                    158                 1                    0.990
   149                    158                 1                    0.993
   150                    158                 1                    0.995
   

## 7. Stream Validated

The live stream correctly reproduces the notebook's core finding: near-zero
maintenance probability during early cycles, rising sharply and reliably in the
final ~15-20% of an engine's life (probability climbs from 0.70 to 0.99 in the last
20 cycles of this example). This confirms the synthetic stream is behaviorally
consistent with the offline evaluation — no surprises introduced by the generator
or the windowed feature calculation in `tick()`.

This stream is ready to feed the dashboard's live chart and risk panel.

## 8. Save as a Reusable Module

In [29]:
SYNTHETIC_DIR = os.path.join(PROJECT_ROOT, 'src', 'synthetic')
os.makedirs(SYNTHETIC_DIR, exist_ok=True)

module_code = '''
"""Synthetic C-MAPSS engine generator and live streaming interface."""
import numpy as np
import pandas as pd


def generate_synthetic_engine(train_df, sensors, life_stretch_std=0.10, noise_std=0.02, seed=None):
    rng = np.random.default_rng(seed)
    template_unit = rng.choice(train_df["unit_number"].unique())
    template = train_df[train_df["unit_number"] == template_unit].sort_values("time_in_cycles").reset_index(drop=True)

    stretch = np.clip(1.0 + rng.normal(0, life_stretch_std), 0.6, 1.6)
    n_cycles = max(int(len(template) * stretch), 30)

    old_idx = np.linspace(0, len(template) - 1, len(template))
    new_idx = np.linspace(0, len(template) - 1, n_cycles)

    synth = pd.DataFrame({"time_in_cycles": np.arange(1, n_cycles + 1)})
    for s in sensors:
        interpolated = np.interp(new_idx, old_idx, template[s].values)
        noise = rng.normal(0, noise_std * (template[s].std() + 1e-6), size=n_cycles)
        synth[s] = interpolated + noise
    return synth, template_unit


def make_plausibility_filter(train_df, sensors):
    real_lifespans = train_df.groupby("unit_number")["time_in_cycles"].max()
    life_min, life_max = real_lifespans.min() * 0.8, real_lifespans.max() * 1.2
    real_ranges = {s: (train_df[s].min(), train_df[s].max()) for s in sensors}

    def is_plausible(synth_df):
        if not (life_min <= len(synth_df) <= life_max):
            return False
        for s in sensors:
            lo, hi = real_ranges[s]
            margin = (hi - lo) * 0.3
            if synth_df[s].min() < lo - margin or synth_df[s].max() > hi + margin:
                return False
        return True

    return is_plausible


class SyntheticEngineStream:
    """Stateful, tick-based synthetic engine stream with live model inference."""

    def __init__(self, train_df, sensors, feature_cols, model, seed=0):
        self.train_df = train_df
        self.sensors = sensors
        self.feature_cols = feature_cols
        self.model = model
        self.rng_seed = seed
        self.is_plausible = make_plausibility_filter(train_df, sensors)
        self._start_new_engine()

    def _start_new_engine(self):
        while True:
            synth, src = generate_synthetic_engine(self.train_df, self.sensors, seed=self.rng_seed)
            self.rng_seed += 1
            if self.is_plausible(synth):
                self.current_engine = synth
                self.current_source = src
                self.cycle_idx = 0
                break

    def tick(self):
        if self.cycle_idx >= len(self.current_engine):
            self._start_new_engine()

        window = self.current_engine.iloc[: self.cycle_idx + 1]
        row = window.iloc[-1].to_dict()

        feats = {}
        for s in self.sensors:
            vals = window[s].values
            cycles = window["time_in_cycles"].values
            feats[s] = vals[-1]
            feats[f"{s}_rm"] = vals[-10:].mean()
            feats[f"{s}_dev"] = vals[-1] - vals[0]
            feats[f"{s}_trend"] = np.polyfit(cycles, vals, 1)[0] if len(vals) > 1 else 0.0

        X = pd.DataFrame([feats])[self.feature_cols]
        proba = self.model.predict_proba(X)[0, 1]
        pred = int(proba >= 0.5)

        self.cycle_idx += 1

        return {
            "engine_source_unit": int(self.current_source),
            "cycle": int(row["time_in_cycles"]),
            "engine_total_lifespan": len(self.current_engine),
            "sensors": {s: round(row[s], 3) for s in self.sensors},
            "maintenance_flag": pred,
            "maintenance_probability": round(float(proba), 3),
        }
'''

with open(os.path.join(SYNTHETIC_DIR, 'cmapss_generator.py'), 'w') as f:
    f.write(module_code)

print("Saved:", os.path.join(SYNTHETIC_DIR, 'cmapss_generator.py'))

Saved: /Users/erenosma/Downloads/predictive-maintenance-dashboard/predictive-maintenance-dashboard/src/synthetic/cmapss_generator.py
